T5 PubMed Summarizer
Transformer-based medical text summarization using the PubMed dataset
Fine-tuned T5-small to generate concise summaries of biomedical research abstracts.

Import Libraries and Load Dataset

In [5]:
import pandas as pd
from datasets import Dataset

# Load CSV
df = pd.read_csv("pubmed_train.csv")

# Drop rows with missing article or abstract
df = df.dropna(subset=["article", "abstract"])

#  remove very short or empty rows 
df = df[df["article"].str.strip() != ""]
df = df[df["abstract"].str.strip() != ""]

# Reset index
df = df.reset_index(drop=True)

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Use smaller subset for CPU speed
dataset = dataset.select(range(min(1000, len(dataset))))

print(" Cleaned dataset loaded successfully!")
print(dataset)



✅ Cleaned dataset loaded successfully!
Dataset({
    features: ['article', 'abstract'],
    num_rows: 1000
})


In [6]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Preprocess & Tokenize

In [7]:
## Preprocessing function

def preprocess_function(examples):
    inputs = ["summarize: " + str(doc) for doc in examples["article"]]  # ensure str()
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["abstract"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

## Tokenize dataset
tokenized_data = dataset.map(preprocess_function, batched=True, remove_columns=["article", "abstract"])



Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

C:\Users\User\AppData\Roaming\Python\Python310\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [8]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
import evaluate
import numpy as np

# Data collator helps batch sequences of different lengths
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Load ROUGE metric for summarization
rouge = evaluate.load("rouge")

# computing evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v * 100, 2) for k, v in result.items()}


In [ ]:
Train, Test & Save the Model

In [10]:
#Define Training Parameters

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="no",      
    learning_rate=3e-5,
    per_device_train_batch_size=2, # small batch size for CPU
    num_train_epochs=1,           
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=1,
)


In [11]:
#Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

C:\Users\User\AppData\Local\Temp\ipykernel_17776\2715366972.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
#Train the Model
trainer.train()

C:\Users\User\AppData\Roaming\Python\Python310\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,3.382400
200,3.069800
300,2.990600
400,2.882800
500,2.942700


TrainOutput(global_step=500, training_loss=3.0536768798828127, metrics={'train_runtime': 3990.8138, 'train_samples_per_second': 0.251, 'train_steps_per_second': 0.125, 'total_flos': 135341801472000.0, 'train_loss': 3.0536768798828127, 'epoch': 1.0})

In [13]:
# Pick one sample to test
sample = df.iloc[0]["article"]

# Tokenize input
inputs = tokenizer("summarize: " + sample, return_tensors="pt", truncation=True, max_length=512)

# Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    min_length=40,
    num_beams=4,
    early_stopping=True
)

# Decode
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Display
print(" Original Article:\n", sample[:500], "...\n")
print(" Generated Summary:\n", summary)


 Original Article:
 a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . 
 in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . 
 the prevalence of malnutrition among elementary school aged ch ...

 Generated Summary:
 in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2%, 8.6% and 1.5%, respectively. snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day.


In [14]:
model.save_pretrained("./t5_pubmed_model")
tokenizer.save_pretrained("./t5_pubmed_model")

print(" Model and tokenizer saved successfully!")

 Model and tokenizer saved successfully!
